# Notebook 05 — Thesis Appendices (B–D) and supplementary outputs



**Printed thesis:** Appendix A (administrative form) appears only in the PDF. This notebook starts with **Appendix B** (model selection and GARCH diagnostics), then **Appendix C** (coefficient estimates), **Appendix D** (data characteristics).



Further sections (**supplementary**) repeat stepwise ARIMA diagnostics, full AIC grids, residual plots, rolling OOS figures, and Ridge/Lasso tables — useful for replication but not necessarily all in the printed appendix.



Uses processed CSVs under `DXY_part/` and `EMCI_part/`. Run **01 → 02 → 03** in each part first; run this notebook with the **repository root** as the working directory.



In [ ]:


import warnings

from pathlib import Path



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import pmdarima as pm



from arch import arch_model

from sklearn.linear_model import RidgeCV, LassoCV

from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import StandardScaler

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsmodels.tsa.statespace.sarimax import SARIMAX

from statsmodels.tsa.stattools import adfuller, kpss



warnings.filterwarnings("ignore")



ROOT = Path.cwd().resolve()

if (ROOT / "DXY_part").is_dir():

    pass

elif (ROOT.parent / "DXY_part").is_dir():

    ROOT = ROOT.parent

elif ROOT.name == "notebooks" and (ROOT.parent.parent / "DXY_part").is_dir():

    ROOT = ROOT.parent.parent

elif ROOT.name == "notebooks" and (ROOT.parent / "DXY_part").is_dir():

    ROOT = ROOT.parent



DXY = ROOT / "DXY_part"

EMCI = ROOT / "EMCI_part"

OUT_TABLES = ROOT / "reports" / "tables"

OUT_FIGS = ROOT / "reports" / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)

OUT_FIGS.mkdir(parents=True, exist_ok=True)



plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({

    "figure.figsize": (14, 8),

    "font.size": 13,

    "axes.titlesize": 15,

    "axes.labelsize": 13,

    "legend.fontsize": 11,

})



RANDOM_SEED = 42

REFIT_EVERY = 21



print(f"Project root: {ROOT}")

print(f"Tables out:   {OUT_TABLES}")

print(f"Figures out:  {OUT_FIGS}")



In [ ]:


DXY_EXOG = ["EEM", "HG=F", "^TNX", "GC=F", "CL=F", "EURUSD=X", "GBPUSD=X", "CAD=X", "^GSPC"]

EEM_EXOG = ["DX-Y.NYB", "^VIX", "^TNX", "CL=F", "HG=F", "GC=F", "^GSPC", "CAD=X", "JPY=X"]



# Joint AR(p)+GARCH orders matching NB02 AIC selection (see 02_model_fitting)

JOINT_AR_LAGS = {"DXY": 0, "EEM": 5}



PANEL_SPECS = {

    "DXY": {

        "root": DXY,

        "feature_csv": DXY / "data" / "processed" / "dxy_dataset_features.csv",

        "target_csv": DXY / "data" / "processed" / "dxy_dataset_target.csv",

        "exog_cols": DXY_EXOG,

        "arima_order": (0, 0, 0),

        "arimax_order": (0, 0, 1),

    },

    "EEM": {

        "root": EMCI,

        "feature_csv": EMCI / "data" / "processed" / "eem_dataset_features.csv",

        "target_csv": EMCI / "data" / "processed" / "eem_dataset_target.csv",

        "exog_cols": EEM_EXOG,

        "arima_order": (2, 0, 1),

        "arimax_order": (2, 0, 1),

    },

}





def load_processed_panel(asset: str) -> dict:

    spec = PANEL_SPECS[asset]

    X_full = pd.read_csv(spec["feature_csv"], index_col=0, parse_dates=True)

    y_df = pd.read_csv(spec["target_csv"], index_col=0, parse_dates=True)

    y = y_df.iloc[:, 0].astype(float)



    lag_cols = [c for c in X_full.columns if "target" in c.lower() and "lag" in c.lower()]

    exog_cols = [c for c in spec["exog_cols"] if c in X_full.columns]



    X = X_full[lag_cols + exog_cols].copy()

    X["vol_21"] = y.rolling(21).std()

    X["vol_63"] = y.rolling(63).std()

    X = X.dropna()

    y = y.loc[X.index]



    shift_cols = exog_cols + ["vol_21", "vol_63"]

    X[shift_cols] = X[shift_cols].shift(1)

    X = X.dropna()

    y = y.loc[X.index]



    train_size = int(len(y) * 0.8)

    return {

        "asset": asset,

        "spec": spec,

        "X_full": X_full,

        "target_full": y_df.iloc[:, 0].astype(float),

        "X": X,

        "y": y,

        "exog": X[exog_cols],

        "exog_cols": exog_cols,

        "lag_cols": lag_cols,

        "train_size": train_size,

        "y_train": y.iloc[:train_size],

        "X_train": X.iloc[:train_size],

        "exog_train": X[exog_cols].iloc[:train_size],

        "y_test": y.iloc[train_size:],

    }





def fit_sarimax(y, order, exog=None):

    exog_data = None if exog is None else exog.loc[y.index]

    model = SARIMAX(

        y,

        exog=exog_data,

        order=order,

        trend="c",

        enforce_stationarity=False,

        enforce_invertibility=False,

    )

    return model.fit(disp=False, method="lbfgs")





def fit_twostep_garch(y_train, order, exog_train=None):

    sarimax_res = fit_sarimax(y_train, order=order, exog=exog_train)

    resid = pd.Series(sarimax_res.resid, index=y_train.index[-len(sarimax_res.resid):]).dropna()

    garch_res = arch_model(

        resid * 100,

        mean="Zero",

        vol="GARCH",

        p=1,

        q=1,

        dist="t",

    ).fit(disp="off", show_warning=False)

    return sarimax_res, garch_res





def result_table_from_sarimax(res, asset, model_name):

    names = list(res.params.index)

    return pd.DataFrame({

        "asset": asset,

        "model": model_name,

        "parameter": names,

        "estimate": res.params.values,

        "std_error": res.bse.values,

        "t_stat": res.tvalues.values,

        "p_value": res.pvalues.values,

    })





def save_table(df, filename):

    path = OUT_TABLES / filename

    df.to_csv(path, index=False)

    print(f"Saved: {path}")

    return path





def fit_joint_ar_garch_on_train(panel, p: int):

    y_s = panel["y_train"].dropna() * 100

    if p == 0:

        model = arch_model(y_s, mean="Constant", vol="GARCH", p=1, q=1, dist="t")

    else:

        model = arch_model(y_s, mean="AR", lags=p, vol="GARCH", p=1, q=1, dist="t")

    return model.fit(disp="off", show_warning=False)





def fit_joint_arx_garch(panel, ar_lags=5):

    y_s = panel["y_train"].dropna() * 100

    exog_fit = panel["exog_train"].loc[y_s.index]

    model = arch_model(

        y_s,

        mean="ARX",

        lags=ar_lags,

        x=exog_fit,

        vol="GARCH",

        p=1,

        q=1,

        dist="t",

    )

    return model.fit(disp="off", show_warning=False)





def garch_vol_rows_from_arch_result(fitted, asset, mean_label):

    rows = []

    for param in ["omega", "alpha[1]", "beta[1]", "nu"]:

        rows.append({

            "asset": asset,

            "mean_model": mean_label,

            "parameter": param,

            "estimate": float(fitted.params.get(param, np.nan)),

            "std_error": float(fitted.std_err.get(param, np.nan)),

            "t_stat": float(fitted.tvalues.get(param, np.nan)),

            "p_value": float(fitted.pvalues.get(param, np.nan)),

        })

    return pd.DataFrame(rows)





def joint_arx_garch_parameter_table(res, asset, ar_lags=5):

    params = pd.Series(res.params)

    rows = []

    for parameter in params.index:

        if parameter in {"omega", "alpha[1]", "beta[1]"}:

            component = "volatility"

        elif parameter == "nu":

            component = "distribution"

        else:

            component = "mean"



        if component == "mean" and not parameter.startswith("target["):

            scale_factor = 100.0

            reported_units = "raw_return"

        elif parameter == "omega":

            scale_factor = 10_000.0

            reported_units = "raw_return_variance"

        else:

            scale_factor = 1.0

            reported_units = "dimensionless"



        rows.append({

            "asset": asset,

            "model": f"Joint ARX({ar_lags})+GARCH(1,1)",

            "component": component,

            "parameter": parameter,

            "estimate": float(res.params[parameter]) / scale_factor,

            "std_error": float(res.std_err[parameter]) / scale_factor,

            "t_stat": float(res.tvalues[parameter]),

            "p_value": float(res.pvalues[parameter]),

            "reported_units": reported_units,

        })

    return pd.DataFrame(rows)





panels = {asset: load_processed_panel(asset) for asset in PANEL_SPECS}

for asset, panel in panels.items():

    print(f"{asset}: modelling rows={len(panel['y'])}, train={panel['train_size']}, test={len(panel['y']) - panel['train_size']}, exog={panel['exog_cols']}")



## Appendix B — Model Selection and Diagnostics



### B.1 GARCH(1,1) parameter estimates — Table B.1 (two-step ARIMA+GARCH)



Refits the two-step mean equation on the NB02 training split and reports Student-t GARCH(1,1) parameters on ARIMA residuals scaled by 100 (`arch`).


In [ ]:


twostep_fits = {}

garch_rows = []



for asset, panel in panels.items():

    order = panel["spec"]["arima_order"]

    sarimax_res, garch_res = fit_twostep_garch(panel["y_train"], order=order)

    twostep_fits[(asset, "ARIMA+GARCH")] = {"sarimax": sarimax_res, "garch": garch_res}



    for param in ["omega", "alpha[1]", "beta[1]", "nu"]:

        garch_rows.append({

            "asset": asset,

            "table": "B.1",

            "mean_model_order": str(order),

            "parameter": param,

            "estimate": float(garch_res.params.get(param, np.nan)),

            "std_error": float(garch_res.std_err.get(param, np.nan)),

            "t_stat": float(garch_res.tvalues.get(param, np.nan)),

            "p_value": float(garch_res.pvalues.get(param, np.nan)),

        })



garch_b1 = pd.DataFrame(garch_rows)

garch_b1["estimate_se"] = garch_b1.apply(

    lambda r: f"{r['estimate']:.6f} ({r['std_error']:.6f})", axis=1

)

garch_b1_wide = garch_b1.pivot(index="parameter", columns="asset", values="estimate_se").reset_index()



save_table(garch_b1, "appendix_b1_twostep_garch_parameter_estimates_long.csv")

save_table(garch_b1_wide, "appendix_b1_twostep_garch_parameter_estimates_wide.csv")

# Backward-compatible aliases

save_table(garch_b1.drop(columns=["table"], errors="ignore"), "appendix_c_garch_parameter_estimates_long.csv")

save_table(garch_b1_wide, "appendix_c_garch_parameter_estimates_wide.csv")

garch_b1_wide



### Table B.2 — Joint AR(p)+GARCH(1,1) parameter estimates



Joint maximum likelihood with Student-t innovations; returns scaled by 100 in estimation. Orders $p$ match NB02 (`JOINT_AR_LAGS`).


In [ ]:


rows_b2 = []

for asset, panel in panels.items():

    p = JOINT_AR_LAGS[asset]

    res = fit_joint_ar_garch_on_train(panel, p)

    label = f"AR({p})+GARCH(1,1)"

    sub = garch_vol_rows_from_arch_result(res, asset, label)

    sub["table"] = "B.2"

    rows_b2.append(sub)



garch_b2_long = pd.concat(rows_b2, ignore_index=True)

garch_b2_long["estimate_se"] = garch_b2_long.apply(

    lambda r: f"{r['estimate']:.6f} ({r['std_error']:.6f})", axis=1

)

garch_b2_wide = garch_b2_long.pivot_table(

    index="parameter", columns=["asset", "mean_model"], values="estimate_se", aggfunc="first"

)

save_table(garch_b2_long, "appendix_b2_joint_ar_garch_parameter_estimates_long.csv")

save_table(garch_b2_wide.reset_index(), "appendix_b2_joint_ar_garch_parameter_estimates_wide.csv")

garch_b2_long



### Table B.3 — Joint ARX(5)+GARCH(1,1) volatility parameters



Variance and tail parameters from the same joint specification used for OOS in NB02 (mean equation coefficients appear under Appendix C).


In [ ]:


JOINT_ARX_AR_LAGS = 5

joint_arx_fits = {}

rows_b3 = []



for asset, panel in panels.items():

    res = fit_joint_arx_garch(panel, ar_lags=JOINT_ARX_AR_LAGS)

    joint_arx_fits[asset] = res

    label = f"ARX({JOINT_ARX_AR_LAGS})+GARCH(1,1)"

    sub = garch_vol_rows_from_arch_result(res, asset, label)

    sub["table"] = "B.3"

    rows_b3.append(sub)



garch_b3_long = pd.concat(rows_b3, ignore_index=True)

garch_b3_long["estimate_se"] = garch_b3_long.apply(

    lambda r: f"{r['estimate']:.6f} ({r['std_error']:.6f})", axis=1

)

save_table(garch_b3_long, "appendix_b3_joint_arx_garch_volatility_estimates_long.csv")

garch_b3_long



### B.2 ARX–GARCH model selection — Table B.4



Joint ARX($p$)+GARCH(1,1) AIC for $p=0,\ldots,5$; $\Delta$AIC vs the best model within each asset (thesis-style top block). Full grid (through $p=8$) is saved under *Supplementary* outputs.


In [ ]:


def joint_arx_garch_grid(panel, max_lags=5):

    y_s = panel["y_train"].dropna() * 100

    exog_fit = panel["exog_train"].loc[y_s.index]

    rows = []

    for p in range(max_lags + 1):

        try:

            kwargs = {"x": exog_fit}

            if p > 0:

                kwargs["lags"] = p

            model = arch_model(

                y_s,

                mean="ARX",

                vol="GARCH",

                p=1,

                q=1,

                dist="t",

                **kwargs,

            )

            res = model.fit(disp="off", show_warning=False)

            rows.append({

                "asset": panel["asset"],

                "model": f"ARX({p})+GARCH(1,1)",

                "ar_lag": p,

                "aic": float(res.aic),

                "delta_aic": np.nan,

            })

        except Exception as exc:

            rows.append({

                "asset": panel["asset"],

                "model": f"ARX({p})+GARCH(1,1)",

                "ar_lag": p,

                "aic": np.nan,

                "delta_aic": np.nan,

                "error": repr(exc),

            })

    return pd.DataFrame(rows)





def compact_b4_table(grid_all):

    frames = []

    for asset in grid_all["asset"].unique():

        g = grid_all[(grid_all["asset"] == asset) & grid_all["aic"].notna()].copy()

        if g.empty:

            continue

        best = g["aic"].min()

        g["delta_aic"] = g["aic"] - best

        g = g.nsmallest(3, "aic")

        frames.append(g)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()





grid_b4 = pd.concat([joint_arx_garch_grid(panel, max_lags=5) for panel in panels.values()], ignore_index=True)

table_b4 = compact_b4_table(grid_b4)

save_table(table_b4, "appendix_b4_joint_arx_garch_aic_ranking_top3.csv")

save_table(grid_b4, "appendix_b4_joint_arx_garch_aic_grid_p0_p5.csv")

table_b4



## Appendix C — Coefficient Estimates



### C.1 Table C.1 — Two-step ARIMAX mean equation (coefficients)



SARIMAX estimates for the ARIMAX orders used in NB02 (two-step procedure; GARCH stage separate).


In [ ]:


arimax_fits = {}

arimax_coef_tables = []



for asset, panel in panels.items():

    order = panel["spec"]["arimax_order"]

    res = fit_sarimax(panel["y_train"], order=order, exog=panel["exog_train"])

    arimax_fits[asset] = res

    arimax_coef_tables.append(result_table_from_sarimax(res, asset=asset, model_name=f"ARIMAX{order}"))



arimax_coefficients = pd.concat(arimax_coef_tables, ignore_index=True)

save_table(arimax_coefficients, "appendix_c1_arimax_coefficient_estimates.csv")

save_table(arimax_coefficients, "appendix_g_arimax_coefficient_estimates.csv")

arimax_coefficients



### C.2 Table C.2 — Joint ARX(5)+GARCH(1,1) full parameter table



Uses the fits from **Table B.3** (`joint_arx_fits`). Mean-equation coefficients are scaled back to raw-return units where noted.


In [ ]:


joint_arx_coef_tables = [

    joint_arx_garch_parameter_table(joint_arx_fits[asset], asset=asset, ar_lags=JOINT_ARX_AR_LAGS)

    for asset in PANEL_SPECS

]

joint_arx_coefficients = pd.concat(joint_arx_coef_tables, ignore_index=True)

save_table(joint_arx_coefficients, "appendix_c2_joint_arx_garch_coefficient_estimates.csv")

save_table(joint_arx_coefficients, "appendix_g2_joint_arx_garch_coefficient_estimates.csv")

joint_arx_coefficients



## Appendix D — Data Characteristics



### D.1 Table D.1 — ADF and KPSS (exogenous log-returns)



Panel-long file keeps both DXY and EEM panels; **thesis-style** summary deduplicates by canonical predictor name (one row per series).


In [ ]:


FEATURE_TICKER_LABELS = {

    "^VIX": "VIX",

    "^TNX": "TNX",

    "^GSPC": "S&P 500",

    "CL=F": "WTI Oil",

    "HG=F": "Copper",

    "GC=F": "Gold",

    "GBPUSD=X": "GBP/USD",

    "EURUSD=X": "EUR/USD",

    "JPY=X": "USD/JPY",

    "CAD=X": "USD/CAD",

    "DX-Y.NYB": "DXY",

    "EEM": "EEM",

}





def canonical_name(ticker: str) -> str:

    return FEATURE_TICKER_LABELS.get(ticker, ticker)





stationarity_rows = []



for asset, panel in panels.items():

    for col in panel["exog_cols"]:

        s = panel["exog"][col].dropna()

        adf_stat, adf_p, adf_lag, adf_nobs, adf_cv, adf_ic = adfuller(s, regression="c", autolag="AIC")

        kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(s, regression="c", nlags="auto")

        stationarity_rows.append({

            "asset_panel": asset,

            "variable": col,

            "canonical": canonical_name(col),

            "nobs": int(len(s)),

            "adf_stat": float(adf_stat),

            "adf_p_value": float(adf_p),

            "adf_lags": int(adf_lag),

            "adf_stationary_5pct": bool(adf_p < 0.05),

            "kpss_stat": float(kpss_stat),

            "kpss_p_value": float(kpss_p),

            "kpss_lags": int(kpss_lags),

            "kpss_stationary_5pct": bool(kpss_p >= 0.05),

        })



stationarity_table = pd.DataFrame(stationarity_rows)

stationarity_table["joint_conclusion"] = np.where(

    stationarity_table["adf_stationary_5pct"] & stationarity_table["kpss_stationary_5pct"],

    "Stationary by ADF and KPSS",

    "Mixed / review",

)



save_table(stationarity_table, "appendix_d1_adf_kpss_exogenous_returns_by_panel.csv")

save_table(stationarity_table, "appendix_d_adf_kpss_exogenous_returns.csv")



# Thesis-style: one row per canonical series (first panel occurrence wins; series are identical)

st_d1_thesis = (

    stationarity_table.sort_values(["canonical", "asset_panel"])

    .groupby("canonical", as_index=False)

    .first()

)

save_table(st_d1_thesis, "appendix_d1_adf_kpss_exogenous_thesis_summary.csv")

stationarity_table.head(10)



### D.2 Table D.2 — Descriptive statistics for exogenous predictors (% log-returns)



Aligned modelling sample; panel-long and deduplicated **thesis summary** with skewness and Min/Max.


In [ ]:


desc_rows = []

for asset, panel in panels.items():

    for col in panel["exog_cols"]:

        s = panel["exog"][col].dropna()

        desc_rows.append({

            "asset_panel": asset,

            "variable": col,

            "canonical": canonical_name(col),

            "nobs": int(len(s)),

            "mean": float(s.mean()),

            "std": float(s.std(ddof=1)),

            "skewness": float(s.skew()),

            "kurtosis": float(s.kurtosis()),

            "min": float(s.min()),

            "max": float(s.max()),

        })



exog_descriptives = pd.DataFrame(desc_rows)

save_table(exog_descriptives, "appendix_d2_exogenous_descriptive_statistics_by_panel.csv")

save_table(exog_descriptives, "appendix_f_exogenous_descriptive_statistics.csv")





# Table D.2 thesis layout: same predictor order / naming as printed Table D.1 (excludes EEM, DX-Y.NYB;
# shared series use DXY-aligned sample; VIX and JPY from EEM panel).
D2_THESIS_ROWS = [
    ("HG=F", "DXY", "Copper (HG=F)"),
    ("^TNX", "DXY", "10Y Treasury (^TNX)"),
    ("GC=F", "DXY", "Gold (GC=F)"),
    ("CL=F", "DXY", "WTI crude (CL=F)"),
    ("EURUSD=X", "DXY", "EUR/USD (EURUSD=X)"),
    ("GBPUSD=X", "DXY", "GBP/USD (GBPUSD=X)"),
    ("CAD=X", "DXY", "USD/CAD (CAD=X)"),
    ("^GSPC", "DXY", "S&P 500 (^GSPC)"),
    ("^VIX", "EEM", "CBOE Volatility (^VIX)"),
    ("JPY=X", "EEM", "USD/JPY (JPY=X)"),
]


def _d2_thesis_from_descriptives(df):
    rows = []
    for var, panel, label in D2_THESIS_ROWS:
        hit = df[(df["variable"] == var) & (df["asset_panel"] == panel)]
        if hit.empty:
            raise ValueError("D.2 thesis row missing: {!r} @ {!r}".format(var, panel))
        row = hit.iloc[0].to_dict()
        row["thesis_label"] = label
        rows.append(row)
    return pd.DataFrame(rows)


exog_d2_thesis = _d2_thesis_from_descriptives(exog_descriptives)
exog_d2_thesis["min_max"] = exog_d2_thesis.apply(
    lambda r: "{:.2f} / {:.2f}".format(r["min"], r["max"]), axis=1
)
save_table(exog_d2_thesis, "appendix_d2_exogenous_descriptive_thesis_summary.csv")
exog_d2_thesis



---



## Supplementary material (replication)



The following cells are **not** labelled B–D in the printed thesis. They document order selection, extended AIC grids, diagnostics, and ML-related tables.


### Stepwise `auto_arima` (two-step mean orders)



Reproduces `pmdarima.auto_arima(..., stepwise=True)` for ARIMA vs ARIMAX means used in NB02.


In [ ]:


def stepwise_auto_arima_selection(panel, use_exog=False):

    y_train = panel["y_train"].dropna()

    exog_train = panel["exog_train"].loc[y_train.index].values if use_exog else None

    expected_order = panel["spec"]["arimax_order" if use_exog else "arima_order"]



    auto = pm.auto_arima(

        y_train,

        X=exog_train,

        max_p=5,

        max_q=5,

        max_d=1,

        information_criterion="aic",

        seasonal=False,

        stepwise=True,

        suppress_warnings=True,

        error_action="ignore",

    )



    return {

        "asset": panel["asset"],

        "specification": "ARIMAX+GARCH mean" if use_exog else "ARIMA+GARCH mean",

        "uses_exog": use_exog,

        "selected_order": str(auto.order),

        "expected_nb02_order": str(expected_order),

        "matches_nb02": auto.order == expected_order,

        "aic": float(auto.aic()),

        "bic": float(auto.bic()),

        "llf": float(auto.arima_res_.llf),

        "estimator": "pmdarima.auto_arima(stepwise=True)",

        "note": "Reproduces NB02/NB03 two-step order selection; not an exhaustive grid.",

    }





stepwise_rows = []

for panel in panels.values():

    stepwise_rows.append(stepwise_auto_arima_selection(panel, use_exog=False))

    stepwise_rows.append(stepwise_auto_arima_selection(panel, use_exog=True))



stepwise_selection = pd.DataFrame(stepwise_rows)

save_table(stepwise_selection, "appendix_e_stepwise_arima_arimax_selection.csv")

stepwise_selection



### Extended AIC tables (two-step ARIMAX components + joint ARX grid up to $p=8$)


In [ ]:


def joint_arx_garch_grid_long(panel, max_lags=6):

    y_s = panel["y_train"].dropna() * 100

    exog_fit = panel["exog_train"].loc[y_s.index]

    rows = []

    for p in range(max_lags + 1):

        try:

            kwargs = {"x": exog_fit}

            if p > 0:

                kwargs["lags"] = p

            model = arch_model(

                y_s,

                mean="ARX",

                vol="GARCH",

                p=1,

                q=1,

                dist="t",

                **kwargs,

            )

            res = model.fit(disp="off", show_warning=False)

            rows.append({

                "asset": panel["asset"],

                "model_family": "Joint ARX-GARCH",

                "component": "joint_mean_variance",

                "order": f"ARX({p})+GARCH(1,1)",

                "ar_lag": p,

                "aic": float(res.aic),

                "bic": float(res.bic),

                "llf": float(res.loglikelihood),

                "selected": False,

                "estimator": "arch.arch_model",

                "note": "single joint Student-t MLE; returns scaled by 100",

                "error": "",

            })

        except Exception as exc:

            rows.append({

                "asset": panel["asset"],

                "model_family": "Joint ARX-GARCH",

                "component": "joint_mean_variance",

                "order": f"ARX({p})+GARCH(1,1)",

                "ar_lag": p,

                "aic": np.nan,

                "bic": np.nan,

                "llf": np.nan,

                "selected": False,

                "estimator": "arch.arch_model",

                "note": "single joint Student-t MLE; returns scaled by 100",

                "error": repr(exc),

            })

    grid = pd.DataFrame(rows)

    if grid["aic"].notna().any():

        best_idx = grid["aic"].idxmin()

        grid.loc[best_idx, "selected"] = True

        grid["delta_aic"] = grid["aic"] - grid.loc[best_idx, "aic"]

    else:

        grid["delta_aic"] = np.nan

    return grid





def twostep_arimax_garch_aic_rows(panel):

    order = panel["spec"]["arimax_order"]

    sarimax_res, garch_res = fit_twostep_garch(

        panel["y_train"],

        order=order,

        exog_train=panel["exog_train"],

    )

    return pd.DataFrame([

        {

            "asset": panel["asset"],

            "model_family": "Two-step ARIMAX+GARCH",

            "component": "mean_ARIMAX",

            "order": f"ARIMAX{order}",

            "ar_lag": order[0],

            "aic": float(sarimax_res.aic),

            "bic": float(sarimax_res.bic),

            "llf": float(sarimax_res.llf),

            "selected": True,

            "estimator": "statsmodels.SARIMAX",

            "note": "mean equation likelihood; not directly additive with residual GARCH AIC",

            "error": "",

        },

        {

            "asset": panel["asset"],

            "model_family": "Two-step ARIMAX+GARCH",

            "component": "variance_GARCH_on_ARIMAX_residuals",

            "order": f"GARCH(1,1)-t on ARIMAX{order} residuals",

            "ar_lag": np.nan,

            "aic": float(garch_res.aic),

            "bic": float(garch_res.bic),

            "llf": float(garch_res.loglikelihood),

            "selected": True,

            "estimator": "arch.arch_model",

            "note": "second-stage residual variance likelihood; residuals scaled by 100",

            "error": "",

        },

    ])





garch_aic_tables = []

for panel in panels.values():

    garch_aic_tables.append(twostep_arimax_garch_aic_rows(panel))

    garch_aic_tables.append(joint_arx_garch_grid_long(panel, max_lags=6))



garch_aic = pd.concat(garch_aic_tables, ignore_index=True)

garch_aic["aic_rank_within_family"] = garch_aic.groupby(

    ["asset", "model_family"], dropna=False

)["aic"].rank(method="min")



save_table(garch_aic, "appendix_e2_garch_aic_tables.csv")

garch_aic.sort_values(["asset", "model_family", "aic_rank_within_family", "ar_lag"]).head(16)



### DXY standardized residual ACF / PACF (two-step ARIMA+GARCH)


In [ ]:


dxy_std_resid = pd.Series(twostep_fits[("DXY", "ARIMA+GARCH")]["garch"].std_resid).dropna()



fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(dxy_std_resid, lags=40, ax=axes[0], zero=False)

axes[0].set_title("DXY ARIMA+GARCH standardized residuals — ACF")

plot_pacf(dxy_std_resid, lags=40, ax=axes[1], zero=False, method="ywm")

axes[1].set_title("DXY ARIMA+GARCH standardized residuals — PACF")

for ax in axes:

    ax.set_xlabel("Lag")

plt.tight_layout()

fig_path = OUT_FIGS / "appendix_h_dxy_garch_standardized_residual_acf_pacf.pdf"

plt.savefig(fig_path, bbox_inches="tight")

print(f"Saved: {fig_path}")

plt.show()



### Rolling one-step OOS forecasts (loads saved CSVs from each part)


In [ ]:


PREDICTION_ROWS = [

    ("ARIMA+GARCH", "arima_garch_predictions.csv", "cadetblue"),

    ("ARIMAX+GARCH", "arimax_garch_predictions.csv", "darkorange"),

    ("Joint AR+GARCH", "joint_ar_predictions.csv", "teal"),

    ("Joint ARX+GARCH", "joint_arx_predictions.csv", "indianred"),

    ("Ridge", "ridge_predictions.csv", "forestgreen"),

    ("Lasso", "lasso_predictions.csv", "mediumpurple"),

    ("Random Forest", "rf_predictions.csv", "steelblue"),

    ("Grad. Boosting", "gb_predictions.csv", "tab:green"),

    ("SVR", "svr_predictions.csv", "mediumpurple"),

]





def load_prediction_csv(panel, filename):

    path = panel["spec"]["root"] / "data" / "processed" / filename

    if not path.exists():

        return None

    return pd.read_csv(path, index_col=0, parse_dates=True)





def plot_separate_oos_forecasts(asset, panel):

    forecasts = {}

    actual = None

    for label, filename, color in PREDICTION_ROWS:

        df = load_prediction_csv(panel, filename)

        if df is None:

            continue

        forecasts[label] = (df["predicted"], color)

        if actual is None:

            actual = df["actual"]



    if actual is None:

        raise FileNotFoundError(f"No prediction CSVs found for {asset}")



    fig, ax = plt.subplots(figsize=(16, 7))

    ax.plot(actual.index, actual, color="black", lw=1.2, label="Actual", alpha=0.8)

    for label, (pred, color) in forecasts.items():

        ax.plot(pred.index, pred, lw=1.0, label=label, color=color, alpha=0.8)

    ax.axhline(0, color="grey", lw=0.8, alpha=0.7)

    ax.set_title(f"{asset} — rolling one-step-ahead OOS forecasts")

    ax.set_ylabel("Daily log-return")

    ax.set_xlabel("Date")

    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)

    plt.tight_layout()

    path = OUT_FIGS / f"appendix_i_{asset.lower()}_rolling_oos_forecasts.pdf"

    plt.savefig(path, bbox_inches="tight")

    print(f"Saved: {path}")

    return fig





for asset, panel in panels.items():

    plot_separate_oos_forecasts(asset, panel)

plt.show()



### Ridge / Lasso standardized coefficients


In [ ]:


def fit_ridge_lasso_coefficients(panel):

    X_train = panel["X_train"]

    y_train = panel["y_train"]

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X_train)

    tscv = TimeSeriesSplit(n_splits=5)



    ridge_alphas = np.logspace(np.log10(0.001), np.log10(100_000.0), 100)

    lasso_alphas = np.logspace(np.log10(0.001), np.log10(100.0), 100)



    y_scaler = None

    y_fit = y_train

    target_scaled = False

    if panel["asset"] == "DXY":

        y_scaler = StandardScaler()

        y_fit = y_scaler.fit_transform(np.asarray(y_train).reshape(-1, 1)).ravel()

        target_scaled = True



    ridge = RidgeCV(alphas=ridge_alphas, cv=tscv)

    lasso = LassoCV(alphas=lasso_alphas, cv=tscv, max_iter=10000)

    ridge.fit(X_scaled, y_fit)

    lasso.fit(X_scaled, y_fit)



    coef_table = pd.DataFrame({

        "asset": panel["asset"],

        "feature": X_train.columns,

        "ridge_coefficient": ridge.coef_,

        "ridge_abs_coefficient": np.abs(ridge.coef_),

        "lasso_coefficient": lasso.coef_,

        "lasso_abs_coefficient": np.abs(lasso.coef_),

        "lasso_nonzero": lasso.coef_ != 0,

    })

    coef_table["ridge_alpha"] = float(ridge.alpha_)

    coef_table["lasso_alpha"] = float(lasso.alpha_)

    coef_table["target_scaled"] = target_scaled

    coef_table["coefficient_units"] = "standardized_y" if target_scaled else "raw_y"

    return coef_table





ridge_lasso_coefficients = pd.concat(

    [fit_ridge_lasso_coefficients(panel) for panel in panels.values()],

    ignore_index=True,

)

save_table(ridge_lasso_coefficients, "appendix_j_ridge_lasso_standardized_coefficients.csv")

ridge_lasso_coefficients.sort_values(["asset", "ridge_abs_coefficient"], ascending=[True, False]).head(20)



### Sanity checks


In [ ]:


expected_files = [

    OUT_TABLES / "appendix_b1_twostep_garch_parameter_estimates_long.csv",

    OUT_TABLES / "appendix_b2_joint_ar_garch_parameter_estimates_long.csv",

    OUT_TABLES / "appendix_b3_joint_arx_garch_volatility_estimates_long.csv",

    OUT_TABLES / "appendix_b4_joint_arx_garch_aic_ranking_top3.csv",

    OUT_TABLES / "appendix_c1_arimax_coefficient_estimates.csv",

    OUT_TABLES / "appendix_c2_joint_arx_garch_coefficient_estimates.csv",

    OUT_TABLES / "appendix_d1_adf_kpss_exogenous_returns_by_panel.csv",

    OUT_TABLES / "appendix_d1_adf_kpss_exogenous_thesis_summary.csv",

    OUT_TABLES / "appendix_d2_exogenous_descriptive_statistics_by_panel.csv",

    OUT_TABLES / "appendix_d2_exogenous_descriptive_thesis_summary.csv",

    OUT_TABLES / "appendix_e_stepwise_arima_arimax_selection.csv",

    OUT_TABLES / "appendix_e2_garch_aic_tables.csv",

    OUT_TABLES / "appendix_j_ridge_lasso_standardized_coefficients.csv",

    OUT_FIGS / "appendix_h_dxy_garch_standardized_residual_acf_pacf.pdf",

    OUT_FIGS / "appendix_i_dxy_rolling_oos_forecasts.pdf",

    OUT_FIGS / "appendix_i_eem_rolling_oos_forecasts.pdf",

]



sample_check = pd.DataFrame([

    {"asset": asset, "modelling_rows": len(panel["y"]), "train_size": panel["train_size"], "test_size": len(panel["y"]) - panel["train_size"]}

    for asset, panel in panels.items()

])



file_check = pd.DataFrame({

    "path": [str(p.relative_to(ROOT)) for p in expected_files],

    "exists": [p.exists() for p in expected_files],

})



print("Sample sizes:")

print(sample_check.to_string())

print("\nOutput files:")

print(file_check.to_string())

assert file_check["exists"].all(), "Some appendix outputs are missing. Re-run the notebook from the top."

print("All appendix outputs were generated successfully.")

